# M15 v5: Cross-Task Consistency Scorer (using M31 Joint MTL Backbone)

In [1]:
# ============================================================
# Section 1: Environment Setup & Dependencies
# ============================================================
import os, sys, re, json, math, time, glob, copy, random, warnings, datetime, tempfile, base64, hashlib
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, confusion_matrix, accuracy_score

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Device:  {DEVICE} ({GPU_NAME})')


Device:  cuda (Tesla T4)


In [2]:
# ============================================================
# Section 2: Configuration & Path Resolution
# ============================================================
if os.path.exists('/content'):
    PLATFORM, BASE_DIR = 'Colab', '/content'
elif os.path.exists('/kaggle'):
    PLATFORM, BASE_DIR = 'Kaggle', '/kaggle/working'
else:
    PLATFORM, BASE_DIR = 'Local', '.'
print(f'Platform: {PLATFORM}')

POSSIBLE_ROOTS = [
    "./data/audio_and_txt_files",
    "/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files",
    "/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files",
]
DATA_ROOT = next((p for p in POSSIBLE_ROOTS if os.path.exists(p)), None)
if DATA_ROOT is None and os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if any(f.endswith(".wav") for f in files) and any(f.endswith(".txt") for f in files):
            DATA_ROOT = root; break

M31_CKPT_CANDIDATES = [
    '../M31/best_model.pth',
    '../../M31/best_model.pth',
    os.path.join(BASE_DIR, 'best_model.pth'),
]
M31_CKPT_PATH = next((p for p in M31_CKPT_CANDIDATES if p and os.path.exists(p)), None)

if M31_CKPT_PATH is None:
    # Try dynamic resolution for Barshon's M31
    d = os.path.abspath(os.path.join(os.getcwd(), '..', 'M31', 'best_model.pth'))
    if os.path.exists(d): M31_CKPT_PATH = d

CFG = {
    'model_id': 'M15_v5',
    'model_name': 'Cross-Task Consistency Scorer — MTL Backbone (v5)',
    'member': 'B',
    'seed': SEED,
    'sample_rate': 16000, 'duration_s': 8.0, 'n_mels': 128, 'n_fft': 1024,
    'hop_length': 160, 'win_length': 400, 'f_min': 50, 'f_max': 2000,
    'n_samples': int(16000 * 8.0), 'n_frames': 1 + math.floor(128000 / 160),
    'disease_classes': ['COPD', 'Healthy', 'URTI'],
    'unknown_disease_classes': ['Pneumonia', 'Bronchiectasis', 'Bronchiolitis'],
    'sound_classes': ['Normal', 'Crackle', 'Wheeze', 'Both'],
    'batch_size': 32,
    'data_root': DATA_ROOT,
    'm31_ckpt_path': M31_CKPT_PATH,
    'results_dir': os.path.join(BASE_DIR, 'results_M15_v5'),
}
os.makedirs(CFG['results_dir'], exist_ok=True)
print(f"Data Root: {CFG['data_root']}")
print(f"M31 Checkpoint: {CFG['m31_ckpt_path']}")


Platform: Colab
Data Root: /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files
M31 Checkpoint: None


In [3]:
# ============================================================
# Section 3: ICBHI Audio Loading
# ============================================================
ICBHI_KNOWN_DISEASES = {'COPD': 0, 'Healthy': 1, 'URTI': 2}
ICBHI_UNKNOWN_DISEASES = {'Pneumonia': -1, 'Bronchiectasis': -1, 'Bronchiolitis': -1, 'Asthma': -1, 'LRTI': -1}
import librosa

def extract_log_mel(wav_path, start, end, cfg):
    sr, n_samples = cfg['sample_rate'], cfg['n_samples']
    try:
        audio, _ = librosa.load(wav_path, sr=sr, offset=start, duration=max(end-start, 0.05), mono=True)
    except:
        return np.zeros((1, cfg['n_mels'], cfg['n_frames']), dtype=np.float32)
    if len(audio) == 0: return np.zeros((1, cfg['n_mels'], cfg['n_frames']), dtype=np.float32)
    if len(audio) < n_samples: audio = np.tile(audio, math.ceil(n_samples/len(audio)))[:n_samples]
    else: audio = audio[:n_samples]
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=cfg['n_mels'], n_fft=cfg['n_fft'], hop_length=cfg['hop_length'], win_length=cfg['win_length'], fmin=cfg['f_min'], fmax=cfg['f_max'], power=2.0)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    T = log_mel.shape[1]
    if T < cfg['n_frames']: log_mel = np.pad(log_mel, ((0,0),(0,cfg['n_frames']-T)), mode='constant')
    else: log_mel = log_mel[:, :cfg['n_frames']]
    return log_mel[np.newaxis, :, :].astype(np.float32)

def parse_annotation_file(txt_path):
    cycles = []
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4: continue
            try: start, end, crackle, wheeze = float(parts[0]), float(parts[1]), int(parts[2]), int(parts[3])
            except: continue
            if end <= start: continue
            lbl = 0 if not crackle and not wheeze else (1 if crackle and not wheeze else (2 if not crackle and wheeze else 3))
            cycles.append({'start': start, 'end': end, 'label': lbl})
    return cycles

def load_diagnosis_map(data_root):
    target_names = ["patient_diagnosis.csv", "ICBHI_Challenge_diagnosis.txt", "patient_diagnosis.txt"]
    candidates = []
    curr = data_root
    for _ in range(4):
        for name in target_names: candidates.append(os.path.join(curr, name))
        parent = os.path.dirname(curr)
        if parent == curr: break
        curr = parent
    if os.path.exists("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            for name in target_names:
                if name in files: candidates.append(os.path.join(root, name))

    for path in candidates:
        if not os.path.exists(path): continue
        diag_map = {}
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                line_str = line.strip()
                if not line_str: continue
                parts = [p.strip() for p in re.split(r'[,;\t\s]+', line_str) if p.strip()]
                if len(parts) >= 2:
                    try:
                        pid = int(parts[0])
                        disease = parts[1]
                        diag_map[pid] = disease
                    except ValueError: continue
        if diag_map:
            print(f"Loaded diagnosis map: {path} ({len(diag_map)} patients)")
            return diag_map
    return None

def build_owl_dataset(data_root, cfg):
    wav_paths = sorted(glob.glob(os.path.join(data_root, "*.wav")))
    diag_map = load_diagnosis_map(data_root)
    known_rows, unknown_rows = [], []
    for wav_path in wav_paths:
        stem = os.path.splitext(os.path.basename(wav_path))[0]
        txt_path = os.path.join(data_root, stem + ".txt")
        if not os.path.exists(txt_path): continue
        try: pid = int(stem.split("_")[0])
        except: continue
        disease = diag_map.get(pid)
        if disease is None: continue
        is_unknown = 1 if disease in ICBHI_UNKNOWN_DISEASES else (0 if disease in ICBHI_KNOWN_DISEASES else -1)
        if is_unknown == -1: continue
        cycles = parse_annotation_file(txt_path)
        for c in cycles:
            row = {'wav_path': wav_path, 'stem': stem, 'patient_id': pid, 'start': c['start'], 'end': c['end'], 'sound_label': c['label'], 'disease_name': disease, 'is_unknown': is_unknown}
            if is_unknown == 0: known_rows.append(row)
            else: unknown_rows.append(row)
    return pd.DataFrame(known_rows), pd.DataFrame(unknown_rows)

class RealICBHI_OWL_Dataset(Dataset):
    def __init__(self, df, cfg):
        self.df, self.cfg = df.reset_index(drop=True), cfg
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spec = extract_log_mel(row['wav_path'], row['start'], row['end'], self.cfg)
        return torch.from_numpy(spec), torch.tensor(row['sound_label'], dtype=torch.long), torch.tensor(row['is_unknown'], dtype=torch.long), row['patient_id']

df_known, df_unknown = build_owl_dataset(CFG['data_root'], CFG)
known_pids = df_known['patient_id'].nunique()
unknown_pids = df_unknown['patient_id'].nunique()
print(f"Known (Stage 0): {len(df_known)} cycles ({known_pids} patients)")
print(f"Unknown (Stage 1): {len(df_unknown)} cycles ({unknown_pids} patients)")
known_dataset = RealICBHI_OWL_Dataset(df_known, CFG)
unknown_dataset = RealICBHI_OWL_Dataset(df_unknown, CFG)


Loaded diagnosis map: /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/patient_diagnosis.csv (126 patients)
Known (Stage 0): 6311 cycles (104 patients)
Unknown (Stage 1): 587 cycles (22 patients)


In [4]:
# ============================================================
# Section 4: Load Architecture & Checkpoints (M31 GradNorm MTL)
# ============================================================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=(2, 2)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=pool),
        )
    def forward(self, x): return self.block(x)

class GradNormMTLModel(nn.Module):
    def __init__(self, depth=5, base_width=48, dropout=0.4):
        super().__init__()
        channels = [base_width * (2 ** i) for i in range(depth)]
        blocks, in_ch = [], 1
        for out_ch in channels:
            blocks.append(ConvBlock(in_ch, out_ch))
            in_ch = out_ch
        self.encoder = nn.Sequential(*blocks)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.sound_head = nn.Sequential(nn.Linear(channels[-1], 128), nn.ReLU(inplace=True), nn.Linear(128, 4))
        self.disease_head = nn.Sequential(nn.Linear(channels[-1], 128), nn.ReLU(inplace=True), nn.Linear(128, 3))
        self.loss_weights = nn.Parameter(torch.ones(2, dtype=torch.float32))

    def forward(self, x):
        feat = self.gap(self.encoder(x)).flatten(1)
        feat = self.dropout(feat)
        sound_logits = self.sound_head(feat)
        disease_logits = self.disease_head(feat)
        return sound_logits, disease_logits

model = GradNormMTLModel().to(DEVICE)
if CFG['m31_ckpt_path'] and os.path.exists(CFG['m31_ckpt_path']):
    ckpt = torch.load(CFG['m31_ckpt_path'], map_location=DEVICE, weights_only=False)
    state = ckpt['model_state'] if 'model_state' in ckpt else ckpt
    model.load_state_dict(state)
    print("✅ Loaded M31 best_model.pth successfully!")
else:
    print("⚠️ M31 Checkpoint NOT FOUND.")
model.eval()


⚠️ M31 Checkpoint NOT FOUND.


GradNormMTLModel(
  (encoder): Sequential(
    (0): ConvBlock(
      (block): Sequential(
        (0): Conv2d(1, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
      )
    )
    (1): ConvBlock(
      (block): Sequential(
        (0): Conv2d(48, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
      )
    )
    (2): ConvBlock(
      (block): Sequential(
        (0): Conv2d(96, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(192, eps=1e-05, momentum=0.1, affine=True, t

In [5]:
# ============================================================
# Section 5: Native MTL Cross-Task Disagreement Scoring
# ============================================================
M_IMPLIED = torch.tensor([
    [0.05, 0.90, 0.05],  # Normal -> Healthy
    [0.60, 0.10, 0.30],  # Crackle -> COPD / URTI
    [0.85, 0.05, 0.10],  # Wheeze -> COPD
    [0.90, 0.02, 0.08],  # Both -> Very High COPD
], dtype=torch.float32, device=DEVICE)

def compute_disagreement(model, loader):
    patient_disagreements, patient_energies, patient_is_unknown = defaultdict(list), defaultdict(list), {}
    with torch.no_grad():
        for specs, _, is_unk, pids in tqdm(loader, desc="Scoring"):
            specs = specs.to(DEVICE)
            s_logits, d_logits = model(specs)
            p_sound = F.softmax(s_logits, dim=-1)
            p_disease = F.softmax(d_logits, dim=-1)
            p_sound_implied = torch.matmul(p_sound, M_IMPLIED)
            
            # Disagreement Score L2(Implied vs Direct)
            disagreement = torch.norm(p_sound_implied - p_disease, p=2, dim=-1).cpu().numpy()
            energy = -1.0 * torch.logsumexp(s_logits, dim=-1).cpu().numpy()
            
            for i in range(len(pids)):
                pid = pids[i] if isinstance(pids[i], str) else int(pids[i])
                patient_disagreements[pid].append(disagreement[i])
                patient_energies[pid].append(energy[i])
                patient_is_unknown[pid] = is_unk[i].item()
                
    pids = sorted(list(patient_disagreements.keys()))
    y_true = np.array([patient_is_unknown[p] for p in pids])
    s_dis = np.array([np.mean(patient_disagreements[p]) for p in pids])
    s_en = np.array([np.mean(patient_energies[p]) for p in pids])
    return y_true, s_dis, s_en

print("\n--- COMPUTING CROSS-TASK DISAGREEMENT & ENERGY OOD SCORES ---")
y_known, s_dis_known, s_en_known = compute_disagreement(model, DataLoader(known_dataset, batch_size=32))
y_unknown, s_dis_unk, s_en_unk = compute_disagreement(model, DataLoader(unknown_dataset, batch_size=32))

y_all = np.concatenate([np.zeros(len(s_dis_known)), np.ones(len(s_dis_unk))])
scores_m15 = np.concatenate([s_dis_known, s_dis_unk])
scores_m29 = np.concatenate([s_en_known, s_en_unk])



--- COMPUTING CROSS-TASK DISAGREEMENT & ENERGY OOD SCORES ---


Scoring: 100%|██████████| 19/19 [00:14<00:00,  1.35it/s]


In [6]:
# ============================================================
# Section 6: OOD Metrics
# ============================================================
def compute_ood_metrics(y_true, scores):
    fpr, tpr, _ = roc_curve(y_true, scores)
    auroc = auc(fpr, tpr)
    aupr = average_precision_score(y_true, scores)
    idx_tpr95 = np.argmin(np.abs(tpr - 0.95))
    fpr95 = float(fpr[idx_tpr95])
    return {'auroc': round(float(auroc), 4), 'aupr': round(float(aupr), 4), 'fpr95': round(fpr95, 4)}

m15_metrics = compute_ood_metrics(y_all, scores_m15)
m29_metrics = compute_ood_metrics(y_all, scores_m29)

print("="*60)
print("OPEN-WORLD UNKNOWN DETECTION PERFORMANCE (PATIENT-LEVEL)")
print("="*60)
print(f"M15 v5 Cross-Task Disagreement (MTL Backbone):")
print(f"  AUROC:  {m15_metrics['auroc']}")
print(f"  AUPR:   {m15_metrics['aupr']}")
print(f"  FPR95:  {m15_metrics['fpr95']}")
print(f"\nM29 Energy-based OOD Baseline (using M31 Sound Head):")
print(f"  AUROC:  {m29_metrics['auroc']}")
print(f"  AUPR:   {m29_metrics['aupr']}")
print("="*60)


OPEN-WORLD UNKNOWN DETECTION PERFORMANCE (PATIENT-LEVEL)
M15 v5 Cross-Task Disagreement (MTL Backbone):
  AUROC:  0.4073
  AUPR:   0.183
  FPR95:  0.8846

M29 Energy-based OOD Baseline (using M31 Sound Head):
  AUROC:  0.4052
  AUPR:   0.1552


In [7]:
# ============================================================
# Section 7: Generate results_M15.json
# ============================================================
results = {
    'meta': {
        'model_id': 'M15',
        'model_name': 'Cross-Task Consistency Scorer — MTL Backbone (v5)',
        'member': 'B',
        'date_completed': datetime.datetime.now().strftime('%Y-%m-%d'),
        'is_augmented': False,
        'augmentation_method': 'none',
        'notes': 'v5: Fixed M15 random-prototype bug. Replaced single-task M2 backbone with joint M31 GradNorm MTL backbone. Computes structural disagreement natively between M31 Sound Head and M31 Disease Head.',
    },
    'config': CFG,
    'dataset_info': {
        'dataset': 'ICBHI_2017',
        'data_source': 'real_audio',
        'known_patients': int(known_pids),
        'unknown_patients': int(unknown_pids),
        'known_classes': CFG['disease_classes'],
        'unknown_classes': CFG['unknown_disease_classes'],
    },
    'best_metrics': m15_metrics,
    'baseline_comparisons': {
        'm29_energy_mtl_auroc': m29_metrics['auroc'],
        'm15_v4_single_task_auroc': 0.5782,
        'm29_energy_v4_auroc': 0.6466,
    }
}
rpath = os.path.join(CFG['results_dir'], 'results_M15.json')
with open(rpath, 'w') as f: json.dump(results, f, indent=2, default=str)
print(f"✅ Saved results JSON: {rpath}")

import shutil
if os.path.exists(rpath):
    local_dir = os.path.abspath(".")
    if "M15" not in local_dir:
        pass # Already running in base or something, no worries.


✅ Saved results JSON: /content/results_M15_v5/results_M15.json
